Quick experiment to see which is better at detecting truthful answers

- model outputs
- hs
- supressed activations (Hypothesis this is better)

In [2]:
%reload_ext autoreload
%autoreload 2

In [3]:
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [4]:
from loguru import logger
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset, Dataset, load_from_disk
from einops import rearrange, repeat
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.data import DataCollatorForLanguageModeling

import torch
from torch import Tensor
from torch.nn.functional import (
    binary_cross_entropy_with_logits as bce_with_logits,
)
from torch.nn.functional import (
    cross_entropy,
)
from pathlib import Path
from jaxtyping import Float
from torch import Tensor

import functools
import pandas as pd
import numpy as np

import itertools
from tqdm.auto import tqdm
import random
import json
from tqdm.auto import tqdm

In [5]:
import gc
def clear_mem():
    """
    Clear memory
    """
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    return None
clear_mem()

## Load data

In [6]:
acts_outfile = Path('../data/activation_store/ds_at-QwenQwen3-1.7B-truthfulQA-bool-train-316-90_v2')

f_config = acts_outfile.with_suffix(".json")
config = json.load(open(f_config, 'r'))
model_name = config['model_name']
print(config)

ds_a2 = load_from_disk(acts_outfile).with_format("torch")
ds_a2

{'model_name': 'Qwen/Qwen3-1.7B', 'batch_size': 10, 'max_length': 90, 'split': 'train', 'n_rows': 632}


Dataset({
    features: ['acts-mlp.down_proj', 'acts-self_attn', 'acts-mlp.up_proj', 'loss', 'logits', 'hidden_states', 'attention_mask', 'label', 'llm_ans', 'llm_log_prob_true', 'supr_amounts', 'question', 'input_ids'],
    num_rows: 632
})

In [7]:
act_groups = [c for c in ds_a2.column_names if c.startswith('acts-')]
act_groups

['acts-mlp.down_proj', 'acts-self_attn', 'acts-mlp.up_proj']

In [8]:
for k,v in ds_a2[0].items():
    if hasattr(v, 'shape'):
        print(k, v.shape)
    else:
        print(k, type(v))

acts-mlp.down_proj torch.Size([1, 90, 2048])
acts-self_attn torch.Size([1, 90, 2048])
acts-mlp.up_proj torch.Size([1, 90, 6144])
loss torch.Size([])
logits torch.Size([151936])
hidden_states torch.Size([1, 90, 2048])
attention_mask torch.Size([90])
label torch.Size([])
llm_ans torch.Size([2])
llm_log_prob_true torch.Size([])
supr_amounts torch.Size([13, 1, 2048])
question <class 'NoneType'>
input_ids <class 'NoneType'>


In [9]:
ds_a2

Dataset({
    features: ['acts-mlp.down_proj', 'acts-self_attn', 'acts-mlp.up_proj', 'loss', 'logits', 'hidden_states', 'attention_mask', 'label', 'llm_ans', 'llm_log_prob_true', 'supr_amounts', 'question', 'input_ids'],
    num_rows: 632
})

## Stats

ds_a2